In [6]:
from pathlib import Path
import duckdb

In [14]:
# Initialize data Path & database file
data_path = Path('../data/')
db_file = '../olist_medallion_architecture.db'

# Connect to database file
conn = duckdb.connect(db_file)

### Build bronze layer objects
* Simple views w/o any changes to the underlying raw data

In [9]:
# Terms for removal
remove_set = set(['olist', 'order', 'dataset'])

# List raw Olist CSV file names
for p in data_path.glob('*.csv'):
    # Remove terms from file name
    file_name_list = p.stem.split('_')

    # Simplify table name from file name
    view_name_list = [x for x in file_name_list if x not in remove_set]
    view_name = '_'.join(view_name_list)

    # Special case: 
    if view_name == 'items':
        view_name = 'order_items'

    # Create bronze views
    print(f'Creating bronze_{view_name}')
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_{view_name} AS 
        SELECT * FROM read_csv_auto('{data_path}/{p.name}');
    """)

print(f'\nBronze layer views created...')

Creating bronze_brazil_state_code_ref
Creating bronze_customers
Creating bronze_geolocation
Creating bronze_orders
Creating bronze_order_items
Creating bronze_payments
Creating bronze_reviews
Creating bronze_products
Creating bronze_sellers
Creating bronze_product_category_name_translation

Bronze layer views created...


### Build silver layer views from bronze views
* CAST all date/timestamp columns explicitly as timestamp
* Simplify columns names
* Deduplicate and clean data (NEEDS ATTENTION)
* Join views using primary and foreign key relations
* Translate review text and add

In [10]:
# import os
import torch
import pyarrow as pa
# from dotenv import load_dotenv 
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Get Huggingface token
# load_dotenv()
# HF_TOKEN = os.getenv('HF_TOKEN')

# Load translation model onto GPU (once)
model_name = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang = 'por_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, dtype = torch.float16
).to("cuda")
model.eval()

# Define output language and batch size
ENG_TOKEN_ID = tokenizer.convert_tokens_to_ids("eng_Latn")
GPU_BATCH_SIZE = 32 #64

def translate_port_to_eng_arrow(text_col: pa.Array) -> pa.Array:
    # Convert Arrow array to Python list, replacing NULLs with ""
    raw_texts = [('' if val is None else str(val).strip()) for val in text_col.to_pylist()]
    
    # Deduplicate non-empty strings within the DuckDB vector chunk
    unique_non_empty = sorted({t for t in raw_texts if t}, key = len)
    cache = {'': ''}

    # Translate in batches
    for i in range(0, len(unique_non_empty), GPU_BATCH_SIZE):
        # Slice batch of text samples
        batch = unique_non_empty[i : i + GPU_BATCH_SIZE]

        # Convert input strings to token ID tensors
        inputs = tokenizer(
            batch, return_tensors = 'pt', padding = True, truncation = True
        ).to("cuda")

        # Translation loop
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id = ENG_TOKEN_ID,
                max_new_tokens = 80,
                max_length = None
            )

        # Convert token IDs back into text
        decoded = tokenizer.batch_decode(generated, skip_special_tokens = True)
        
        # Load before and after text into cache dict
        for orig, trans in zip(batch, decoded):
            cache[orig] = trans

    # Map back in original order (replace NULL's w/ empty string '')
    translated = [cache[t] for t in raw_texts]
    return pa.array(translated, type = pa.string())


# Register UDF in duckdb
try:
    conn.remove_function('translate_port_to_eng_arrow')
except Exception:
    pass

conn.create_function(
    'translate_port_to_eng_arrow', 
    translate_port_to_eng_arrow, 
    ['VARCHAR'], 
    'VARCHAR',
    type = 'arrow',
    null_handling = 'special')

c:\Users\nate-\Desktop\Job Searching\Olist Portfolio Project\olist-data-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 512/512 [00:01<00:00, 430.88it/s]


In [ ]:
# Order, payments and reviews view definitions
conn.execute("""
    CREATE OR REPLACE VIEW silver_order_items AS
        SELECT
            order_id,
            product_id,
            seller_id,
            order_item_id AS item_sequence,
            CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_timestamp,
            CAST(price AS DECIMAL(12,2)) AS item_price,
            CAST(freight_value AS DECIMAL(12,2)) AS shipping_cost
        FROM bronze_order_items;
""")
conn.execute("""
    CREATE OR REPLACE VIEW silver_orders AS
        SELECT
            order_id,
            o.customer_id,
            c.customer_unique_id AS repeat_customer_id,
            order_status AS status,
            CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_timestamp,
            CAST(order_approved_at AS TIMESTAMP) AS approved_timestamp,
            CAST(order_delivered_carrier_date AS TIMESTAMP) AS delivered_carrier_timestamp,
            CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_customer_timestamp,
            CAST(order_estimated_delivery_date AS TIMESTAMP) AS estimated_delivery_timestamp
        FROM bronze_orders o
        LEFT JOIN bronze_customers c
            ON o.customer_id = c.customer_id;
""")

conn.execute("""
    CREATE OR REPLACE VIEW silver_payments AS
        SELECT
            order_id,
            payment_sequential AS payment_sequence,
            payment_type AS type,
            payment_installments AS installments,
            CAST(payment_value AS DECIMAL(12,2)) AS value
        FROM bronze_payments;
""")

# (REVIEW) Consider averaging repeated ratings, or at least adding an ORDER BY clause to the ROW_NUMBER analytical function
# * Row number logic: partition by order_id order by comment text to return record that has review text
conn.execute("""
    CREATE OR REPLACE TABLE silver_reviews AS
        WITH cleaned_and_deduped AS (
            SELECT
                review_id,
                order_id,
                CAST(review_creation_date AS TIMESTAMP) AS creation_timestamp,
                CAST(review_answer_timestamp AS TIMESTAMP) AS answer_timestamp,
                review_score AS rating,
                NULLIF(TRIM(review_comment_title), '') AS title_pt,
                NULLIF(TRIM(review_comment_message), '') AS comment_pt,
                ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY 
                    (comment_pt IS NOT NULL) DESC,
                    (title_pt   IS NOT NULL) DESC,
                    answer_timestamp   DESC NULLS LAST,
                    creation_timestamp DESC NULLS LAST,
                    review_id) AS rn
            FROM bronze_reviews
        )
        SELECT
            review_id,
            order_id,
            creation_timestamp,
            answer_timestamp,
            rating,
            title_pt,
            comment_pt,
            translate_port_to_eng_arrow(title_pt) AS title_eng,
            translate_port_to_eng_arrow(comment_pt) AS comment_eng
        FROM cleaned_and_deduped
        WHERE rn = 1;
""")

# Products view definition (w/ english category name)
conn.execute("""
    CREATE OR REPLACE VIEW silver_products AS
        SELECT
            p.product_id,
            REPLACE(COALESCE(pt.product_category_name_english, p.product_category_name), '_', ' ') AS category,
            product_name_lenght AS name_length,
            product_description_lenght AS description_length,
            product_photos_qty AS photo_qty,
            product_weight_g AS weight_g,
            product_length_cm AS length_cm,
            product_height_cm AS height_cm,
            product_width_cm AS width_cm,
            (product_length_cm * product_height_cm * product_width_cm) AS volume_cm3
        FROM bronze_products p
        LEFT JOIN bronze_product_category_name_translation pt 
            ON p.product_category_name = pt.product_category_name;
""")

# Customers & sellers view definitions (w/ formal state name)
conn.execute("""
    CREATE OR REPLACE VIEW silver_customers AS
        SELECT
            customer_id,
            customer_unique_id AS repeat_customer_id,
            customer_zip_code_prefix AS zip_code,
            customer_city AS city,
            customer_state AS state_code,
            bs.state_name
        FROM bronze_customers c
        LEFT JOIN bronze_brazil_state_code_ref bs 
            ON c.customer_state = bs.state_code;
""")

conn.execute("""
    CREATE OR REPLACE VIEW silver_sellers AS
        SELECT
            seller_id,
            seller_zip_code_prefix AS zip_code,
            seller_city AS city,
            seller_state AS state_code,
            bs.state_name
        FROM bronze_sellers s
        LEFT JOIN bronze_brazil_state_code_ref bs 
            ON s.seller_state = bs.state_code;
""")

# Geolocation view definition (w/ lat & long averaged by zip code, city/state removed)
conn.execute("""
    CREATE OR REPLACE VIEW silver_geolocation AS
    SELECT 
        geolocation_zip_code_prefix AS zip_code,
        AVG(geolocation_lat) AS avg_latitude,
        AVG(geolocation_lng) AS avg_longitude
    FROM bronze_geolocation
    GROUP BY
        geolocation_zip_code_prefix;
""")

print(f'\nSilver layer views created...')


Silver layer views created...


### Build gold layer tables from silver views
* Formal tables with primary and foreign keys
* Denormalized views for dashboard reporting

In [ ]:
# Master order-level query to power dashboard
conn.execute("""
    CREATE OR REPLACE TABLE gold_orders AS
        WITH order_items_agg AS (
            SELECT
                order_id,
                MIN(shipping_limit_timestamp) AS min_shipping_limit_timestamp,
                MAX(shipping_limit_timestamp) AS max_shipping_limit_timestamp,
                COUNT(item_sequence) AS total_product_count,
                COUNT(DISTINCT product_id) AS unique_product_count,
                COUNT(DISTINCT seller_id) AS unique_seller_count,
                COALESCE(SUM(item_price), 0) AS total_item_price,
                COALESCE(SUM(shipping_cost), 0) AS total_shipping_cost,
                COALESCE(SUM(item_price + shipping_cost), 0) AS items_total_value
            FROM silver_order_items
            GROUP BY 
                order_id
        ),
    
        main_item AS (   
            -- most expensive line per order
            SELECT
                oi.order_id,
                ARG_MAX(oi.product_id, (oi.item_price, -oi.item_sequence)) AS main_product_id,
                ARG_MAX(COALESCE(p.category, 'unknown'), (oi.item_price, -oi.item_sequence)) AS main_category,
                LIST(DISTINCT COALESCE(p.category, 'unknown') ORDER BY COALESCE(p.category, 'unknown')) AS categories,
                ARG_MAX(oi.seller_id, (oi.item_price, -oi.item_sequence)) AS main_seller_id,
                SUM(p.weight_g) AS total_weight_g,
                SUM(p.volume_cm3) AS total_volume_cm3,
                COUNT(*) FILTER (WHERE p.volume_cm3 IS NULL) items_missing_dims
            FROM silver_order_items oi
            LEFT JOIN silver_products p 
                ON oi.product_id = p.product_id
            GROUP BY 
                oi.order_id
    ),
    
        
        payments_agg AS (
            SELECT
                order_id,
                ARG_MAX(type, (value, -payment_sequence)) AS largest_single_payment_type,
                ARG_MIN(type, payment_sequence) AS initial_payment_type,
                LIST(DISTINCT type ORDER BY type) AS payment_methods_used,
                -- (Mixed payment type categorization)
                CASE
                    WHEN COUNT(DISTINCT type) = 1 AND COUNT(*) > 1
                        AND ANY_VALUE(type) = 'voucher' THEN 'multi_voucher'
                    WHEN COUNT(DISTINCT type) = 1 THEN ANY_VALUE(type)
                    WHEN BOOL_OR(type = 'voucher')
                        AND BOOL_OR(type = 'credit_card') THEN 'mixed_voucher_card'
                    ELSE 'mixed_card'
                END AS payment_category,
                -- (Boolean flag for quick filtering)
                COUNT(DISTINCT type) > 1 AS is_split_payment,
                COUNT(*) AS payment_count,
                MAX(installments) AS max_installments,
                MAX(installments) FILTER (WHERE type = 'credit_card') AS card_installments,
                ROUND(SUM(value), 2) AS payments_total_value
            FROM silver_payments
            GROUP BY 
                order_id
        )
    
        SELECT
            -- order
            o.order_id,
            o.customer_id,
            o.repeat_customer_id,
            o.status,
            o.purchase_timestamp,
            o.approved_timestamp,
            o.delivered_carrier_timestamp,
            o.delivered_customer_timestamp,
            o.estimated_delivery_timestamp,
            DATE_TRUNC('month', o.purchase_timestamp) AS purchase_month,
            DAYNAME(o.purchase_timestamp) AS purchase_weekday,
            HOUR(o.purchase_timestamp) AS purchase_hour,
    
            -- delivery performance
            DATE_DIFF('day', o.purchase_timestamp, o.delivered_customer_timestamp) AS delivery_days,
            DATE_DIFF('day', o.estimated_delivery_timestamp, o.delivered_customer_timestamp) AS days_vs_estimate,
            o.delivered_customer_timestamp > o.estimated_delivery_timestamp AS is_late,
    
            -- customer 
            c.city AS customer_city,
            c.state_code AS customer_state_code,
            c.state_name AS customer_state_name,
    
            -- items
            oi.min_shipping_limit_timestamp,
            oi.max_shipping_limit_timestamp,
            oi.total_product_count,
            oi.unique_product_count,
            oi.unique_seller_count,
            ROUND(oi.total_item_price, 2) AS total_item_price,
            ROUND(oi.total_shipping_cost, 2) AS total_shipping_cost,
            ROUND(oi.items_total_value, 2) AS items_total_value,
    
            -- main product / seller 
            mi.main_product_id,
            mi.main_category,
            mi.categories,
            mi.total_weight_g,
            mi.main_seller_id,
            s.state_code AS main_seller_state_code,
            s.state_code = c.state_code AS is_same_state,
    
            -- payments
            pa.largest_single_payment_type,
            pa.initial_payment_type,
            pa.payment_methods_used,
            pa.payment_category,
            pa.is_split_payment,
            pa.payment_count,
            pa.max_installments,
            pa.card_installments,
            pa.payments_total_value AS payments_total_value,
            ROUND(pa.payments_total_value - oi.items_total_value, 2) AS paid_minus_items,
            ABS(ROUND(pa.payments_total_value - oi.items_total_value, 2)) > 0.01  AS is_payment_mismatch, 
    
            -- review
            r.rating AS review_rating,
            r.comment_pt IS NOT NULL AS has_review_comment,
            r.creation_timestamp AS review_creation_timestamp
        FROM silver_orders o
        LEFT JOIN silver_customers c 
            ON o.customer_id = c.customer_id
        LEFT JOIN order_items_agg oi 
            ON o.order_id = oi.order_id
        LEFT JOIN main_item mi   
            ON o.order_id = mi.order_id
        LEFT JOIN silver_sellers s 
            ON s.seller_id = mi.main_seller_id
        LEFT JOIN payments_agg pa 
            ON o.order_id = pa.order_id
        LEFT JOIN silver_reviews r 
            ON o.order_id = r.order_id;
""")

# Customer and sellers table definitions (w/ averaged lat and long by zip code)
conn.execute("""
    CREATE OR REPLACE TABLE gold_customers AS
        SELECT
            customer_id,
            repeat_customer_id,
            c.zip_code,
            city,
            state_code,
            state_name,
            g.avg_latitude,
            g.avg_longitude
        FROM silver_customers c
        LEFT JOIN silver_geolocation g
            ON c.zip_code = g.zip_code;
""")

conn.execute("""
    CREATE OR REPLACE TABLE gold_sellers AS
        SELECT
            seller_id,
            s.zip_code,
            city,
            state_code,
            state_name,
            g.avg_latitude,
            g.avg_longitude
        FROM silver_sellers s
        LEFT JOIN silver_geolocation g
            ON s.zip_code = g.zip_code;
""")

In [17]:
# Close database connection
conn.close()